# 🛡️ AegisGuard — Fine-Tune Agent-Intent Safety Classifier

**What this does:**
1. Fine-tunes Qwen 2.5 1.5B on 2,919 cloud infrastructure attack examples
2. Exports to ONNX with INT4 quantization
3. Downloads the model (~900MB) for local deployment

**Time:** ~15 minutes on T4 GPU

**Usage:** Click Runtime → Run All, then download the model at the end.

In [ ]:
# Step 1: Install dependencies
!pip install -q transformers datasets peft bitsandbytes accelerate trl
!pip install -q optimum[onnxruntime] onnx onnxruntime

In [ ]:
# Step 2: Download training data from GitHub
import urllib.request
import json

# Download from the repo
url = 'https://raw.githubusercontent.com/yogami/aegis-agent-firewall/main/training-data/aegisguard_train.jsonl'
try:
    urllib.request.urlretrieve(url, 'aegisguard_train.jsonl')
    print('Downloaded training data from GitHub')
except:
    print('GitHub download failed — upload aegisguard_train.jsonl manually')
    from google.colab import files
    uploaded = files.upload()

# Load and inspect
with open('aegisguard_train.jsonl') as f:
    data = [json.loads(line) for line in f]

# Count distribution
from collections import Counter
dist = Counter(d['label'] for d in data)
print(f'\nTotal examples: {len(data)}')
print('\nDistribution:')
for label, count in dist.most_common():
    print(f'  {label:30s} {count:5d}')

In [ ]:
# Step 3: Format data for instruction fine-tuning
from datasets import Dataset

LABEL_LIST = sorted(set(d['label'] for d in data))
LABEL2ID = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

def format_example(example):
    payload = example['payload']
    label = example['label']
    mitre = example.get('mitre') or 'N/A'
    eu_ai = example.get('eu_ai_act') or 'N/A'

    # Format as instruction-response pair
    instruction = f"""Classify this cloud API payload into one of these threat categories: {', '.join(LABEL_LIST)}.

Payload:
Method: {payload.get('method', 'GET')}
Endpoint: {payload.get('endpoint', '/')}
Body: {json.dumps(payload.get('body', {}))}

Respond with ONLY the category name."""

    return {
        'text': f'<|im_start|>user\n{instruction}<|im_end|>\n<|im_start|>assistant\n{label}<|im_end|>',
        'label_id': LABEL2ID[label]
    }

formatted = [format_example(d) for d in data]
dataset = Dataset.from_list(formatted)

# 90/10 train/eval split
split = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split['train']
eval_ds = split['test']

print(f'Train: {len(train_ds)}, Eval: {len(eval_ds)}')
print(f'\nLabels ({len(LABEL_LIST)}): {LABEL_LIST}')
print(f'\nSample:\n{formatted[0]["text"][:500]}')

In [ ]:
# Step 4: Load base model with 4-bit quantization
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)
print(f'Model loaded: {MODEL_ID}')
print(f'Parameters: {model.num_parameters():,}')

In [ ]:
# Step 5: Apply LoRA adapters
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Step 6: Fine-tune with SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./aegisguard-checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy='epoch',
    evaluation_strategy='epoch',
    fp16=True,
    optim='paged_adamw_8bit',
    report_to='none',
    max_grad_norm=0.3,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    dataset_text_field='text',
    max_seq_length=512,
    packing=True,
)

print('Starting training...')
trainer.train()
print('Training complete!')

In [ ]:
# Step 7: Merge LoRA adapters into base model
from peft import PeftModel

# Save the adapter
model.save_pretrained('./aegisguard-lora')
tokenizer.save_pretrained('./aegisguard-lora')

# Reload base model in FP16 and merge
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True
)

merged_model = PeftModel.from_pretrained(base_model, './aegisguard-lora')
merged_model = merged_model.merge_and_unload()

# Save merged model
merged_model.save_pretrained('./aegisguard-merged')
tokenizer.save_pretrained('./aegisguard-merged')

print('Merged model saved to ./aegisguard-merged')

In [ ]:
# Step 8: Quick inference test before export
from transformers import pipeline

pipe = pipeline('text-generation', model=merged_model, tokenizer=tokenizer, max_new_tokens=20)

test_cases = [
    {'method': 'PUT', 'endpoint': '/kms/keys/rotate', 'body': {'snapshot_retention_days': 0}},
    {'method': 'GET', 'endpoint': '/api/v1/users', 'body': {}},
    {'method': 'DELETE', 'endpoint': '/rds/clusters/production', 'body': {'skip_final_snapshot': True}},
    {'method': 'POST', 'endpoint': '/iam/roles/root', 'body': {'policy': 'AdministratorAccess'}},
]

labels = sorted(set(d['label'] for d in data))

print('\n═══ Inference Test ═══')
for tc in test_cases:
    prompt = f"""<|im_start|>user
Classify this cloud API payload into one of these threat categories: {', '.join(labels)}.

Payload:
Method: {tc['method']}
Endpoint: {tc['endpoint']}
Body: {json.dumps(tc['body'])}

Respond with ONLY the category name.<|im_end|>
<|im_start|>assistant
"""
    result = pipe(prompt)[0]['generated_text']
    # Extract just the assistant response
    answer = result.split('<|im_start|>assistant\n')[-1].split('<|im_end|>')[0].strip()
    print(f'  {tc["method"]:6s} {tc["endpoint"]:40s} → {answer}')

In [ ]:
# Step 9: Export to ONNX + INT4 quantization
!optimum-cli export onnx \
    --model ./aegisguard-merged \
    --task text-generation \
    ./aegisguard-onnx/

print('\nONNX export complete!')

In [ ]:
# Step 10: Quantize to INT4
from optimum.onnxruntime import ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

quantizer = ORTQuantizer.from_pretrained('./aegisguard-onnx')
qconfig = AutoQuantizationConfig.avx512_vnni(is_static=False, per_channel=False)

quantizer.quantize(
    save_dir='./aegisguard-onnx-quantized',
    quantization_config=qconfig
)

# Show file sizes
import os
for f in os.listdir('./aegisguard-onnx-quantized'):
    size = os.path.getsize(f'./aegisguard-onnx-quantized/{f}')
    print(f'  {f}: {size / 1024 / 1024:.1f} MB')

In [ ]:
# Step 11: Package for download
import shutil

# Copy essential files
os.makedirs('./aegisguard-release', exist_ok=True)

# Copy quantized ONNX model
for f in os.listdir('./aegisguard-onnx-quantized'):
    shutil.copy2(f'./aegisguard-onnx-quantized/{f}', './aegisguard-release/')

# Copy tokenizer files
for f in ['tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json']:
    src = f'./aegisguard-merged/{f}'
    if os.path.exists(src):
        shutil.copy2(src, './aegisguard-release/')

# Save label map
with open('./aegisguard-release/label_map.json', 'w') as f:
    json.dump({'label2id': LABEL2ID, 'id2label': ID2LABEL, 'labels': LABEL_LIST}, f, indent=2)

# Zip for download
shutil.make_archive('aegisguard-release', 'zip', '.', 'aegisguard-release')

print('\n═══ RELEASE PACKAGE READY ═══')
print(f'Size: {os.path.getsize("aegisguard-release.zip") / 1024 / 1024:.1f} MB')
print('\nDownloading...')

from google.colab import files
files.download('aegisguard-release.zip')

## ✅ Done!

**Next steps:**
1. Unzip `aegisguard-release.zip`
2. Place the contents in `aegis-agent-firewall/backend/src/aegisguard-model/`
3. Push to GitHub → Railway auto-deploys

The V4 integration code is already in the repo waiting for this model.